# Producer Pattern 1 of 5 — Fire-and-Forget

> **This is the baseline.** The other four patterns are all *fire-and-forget plus one extra idea*. Learn this one well and the rest are small steps.

**The one-line idea:** send the message, do **not** wait to find out whether it arrived.

**Real-world picture:** dropping a postcard in a mailbox. You let go and walk away. It almost always arrives. You get **no tracking number**, and if it's lost, nobody tells you.

**Needs:** Notebook 01 (Kafka running), 02 (topics/partitions/offsets), 03 (how `produce`, `poll`, `flush` work). This notebook re-explains the parts that matter, so you can also read it on its own.

| Section | What you learn |
|---|---|
| 0 | Connect to Kafka |
| 1 | The absolute minimum fire-and-forget send |
| 1.5 | **Full mechanics of `produce()`, `poll()`, `flush()` — and producer `poll()` vs consumer `poll()`** |
| 2 | What breaks if you NEVER poll at all: `BufferError` |
| 3 | Why `flush()` at the end is not optional |
| 4 | See the silent failure: a lost message nobody reports |
| 5 | See the payoff: raw speed |
| 6 | Wrap it in a tiny reusable class |
| 7 | When to use it, when never to |
| 8 | What the repo's 800-line version adds, and why that is really *other* patterns |
| 9 | Checkpoint |

## Section 0 — Connect

Run this first. These helpers are the same ones from Notebooks 02–03; the comments recap what each does.

In [1]:
"""
NOTEBOOK HELPERS -- connect to Kafka and manage topics for the demos below.

What this cell does, in order:
1. Connect to our local Kafka broker (AdminClient).
2. Check whether a topic already exists.
3. If it exists, delete it and wait until it is really gone.
4. Create a fresh, empty topic with the requested number of partitions.
5. Print which brokers we are connected to.

A TOPIC is a named stream where messages are stored.
A PARTITION is one section of a topic; a topic can have several, to spread
messages across them (Notebook 02 covers this in depth).

This code is for a LOCAL LEARNING ENVIRONMENT. Do not delete/recreate topics
this carelessly against a production cluster.
"""

import json
import time
from pathlib import Path
import subprocess

from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic


# ============================================================
# STEP 1: CONFIGURATION
# ============================================================

# Address of our Kafka broker. "localhost" = running on this machine;
# 9092 is the port our docker-compose.yml exposes (see Notebook 01).
BOOTSTRAP = "localhost:9092"

# Name of the Docker container running Kafka (see Notebook 01).
CONTAINER = "kafka-lab"

# How many partitions a new topic gets if the caller doesn't say otherwise.
DEFAULT_PARTITIONS = 3


# ============================================================
# STEP 2: CONNECT TO KAFKA
# ============================================================

# AdminClient lets us manage Kafka resources: list, create, delete topics.
# Creating this object does NOT prove Kafka is reachable -- the first real
# request (below) is what actually confirms the connection.
admin = AdminClient({"bootstrap.servers": BOOTSTRAP})


def show_brokers():
    """Ask Kafka for its cluster metadata and print the connected broker(s)."""
    # timeout=10: give up and raise an error if nothing answers within 10s.
    metadata = admin.list_topics(timeout=10)
    for broker in metadata.brokers.values():
        print(f"Connected to Kafka broker: id={broker.id}")


# ============================================================
# STEP 3: CHECK WHETHER A TOPIC EXISTS
# ============================================================

def topic_exists(topic_name):
    """Return True if Kafka already has a topic with this name."""
    metadata = admin.list_topics(timeout=10)
    # metadata.topics is a dict: {"orders": TopicMetadata(...), ...}.
    # We only need the names, i.e. the dictionary's keys.
    return topic_name in metadata.topics


# ============================================================
# STEP 4: DELETE AN EXISTING TOPIC
# ============================================================

def delete_topic(topic_name):
    """Delete a topic (if present) and wait until it has actually disappeared."""
    if not topic_exists(topic_name):
        return     # nothing to do

    # Kafka admin requests are asynchronous: delete_topics() submits the
    # request and returns immediately, WITHOUT waiting for Kafka to finish.
    # What comes back is a dict of FUTURES -- one placeholder per topic we
    # asked to delete, standing in for "the real answer, which arrives later."
    delete_requests = admin.delete_topics([topic_name])

    # Pick out the future for our one topic.
    delete_future = delete_requests[topic_name]

    # .result(timeout=...) blocks until that future resolves: it waits until
    # Kafka confirms the delete really happened, or raises an exception if
    # Kafka reports a failure, or if nothing happens within the timeout.
    delete_future.result(timeout=15)

    # Even after that, Kafka's own topic LISTING can lag a moment behind the
    # real deletion. So we poll "does it still show up?" against a deadline,
    # instead of looping forever if something is stuck.
    deadline = time.monotonic() + 15
    while topic_exists(topic_name):
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Topic '{topic_name}' did not disappear in time.")
        time.sleep(0.5)


# ============================================================
# STEP 5: CREATE A NEW TOPIC
# ============================================================

def create_topic(topic_name, partitions=DEFAULT_PARTITIONS):
    """Create a brand-new topic with the given number of partitions."""
    # NewTopic is just a DESCRIPTION of what we want Kafka to create:
    # name, how many partitions, and replication_factor=1 (one copy of each
    # partition -- fine on our single-broker lab, never in production).
    new_topic = NewTopic(topic_name, num_partitions=partitions, replication_factor=1)

    # Same async pattern as delete_topic(): submit, get back a future per
    # topic, then block on .result() until Kafka confirms creation.
    create_requests = admin.create_topics([new_topic])
    create_future = create_requests[topic_name]
    create_future.result(timeout=15)


# ============================================================
# STEP 6: RECREATE A TOPIC FROM SCRATCH
# ============================================================

def recreate_topic(topic_name, partitions=DEFAULT_PARTITIONS):
    """Replace a topic with a fresh, empty one.

    Why this exists: every demo cell below can call this and know it is
    starting from a brand-new, empty topic, with no leftover messages from
    a previous run.

    WARNING: any messages in the previous topic (if one existed) are gone.
    """
    delete_topic(topic_name)
    create_topic(topic_name, partitions)
    print(f"Topic '{topic_name}' ready with {partitions} partitions.")


# ============================================================
# STEP 7: RUN KAFKA'S OWN COMMAND-LINE TOOLS (used in a few later cells)
# ============================================================

def run_on_host(command, show=True):
    """Run a command on your REAL machine, not inside this sandboxed notebook.

    Why this exists: VS Code here runs as a Flatpak, which sandboxes it away
    from `docker` and other host programs. `flatpak-spawn --host` escapes
    that sandbox. On a normal (non-Flatpak) install, the command just runs
    directly, so this helper is harmless there too.
    """
    in_sandbox = Path("/.flatpak-info").exists()
    full_command = ["flatpak-spawn", "--host", *command] if in_sandbox else command
    result = subprocess.run(full_command, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if output and show:
        print(output)
    return result


def kafka_tool(tool, *args, show=True):
    """Run one of Kafka's own command-line tools inside the kafka-lab container.

    Example: kafka_tool("kafka-topics.sh", "--list", "--bootstrap-server", BOOTSTRAP)
    This is an alternative way to inspect Kafka. Our topic-management functions
    above use AdminClient instead, which is the normal choice from Python code.
    """
    return run_on_host(["docker", "exec", CONTAINER, f"/opt/kafka/bin/{tool}", *args], show=show)


# ============================================================
# STEP 8: VERIFY THE CONNECTION
# ============================================================

show_brokers()

Connected to Kafka broker: id=1


In [2]:
admin.list_topics(timeout=10).topics

{'orders_demo': TopicMetadata(orders_demo, 3 partitions),
 'nb3_acks': TopicMetadata(nb3_acks, 3 partitions),
 'orders': TopicMetadata(orders, 3 partitions)}

## ⚠ There is no "fire-and-forget" setting

This is the single most important thing to understand before reading any code below.

**Fire-and-forget is not a Kafka configuration.** There is no `fire_and_forget=true` you pass in. You will not find it in `acks`, `enable.idempotence`, or anywhere in the broker or client config. **It is a *usage pattern* — a decision about which Python calls you make and don't make.**

Exactly one thing in the code makes a send "fire-and-forget":

```python
producer.produce("ff_demo", value=value_bytes)   # <-- no on_delivery=... argument
```

**It is the *absence* of a delivery callback.** That's it. `produce()` always accepts an optional `on_delivery` function. Leave it out, and you have fire-and-forget: Kafka will still process the message exactly the same way on the broker side, but *nothing in your process is told the outcome*.

Compare the two calls side by side:

```python
# FIRE-AND-FORGET  — no on_delivery
producer.produce("orders", value=data)

# CALLBACK-CONFIRMED (Notebook 2) — same call, one argument added
producer.produce("orders", value=data, on_delivery=my_callback)
```

Nothing else changes. Same topic, same broker, same `acks`, same network trip. The *entire* difference between this pattern and the next notebook's pattern is that one keyword argument.

| What people often assume controls it | Why that's wrong |
|---|---|
| `acks=0` | `acks` controls how many broker replicas must confirm *before the broker answers* — a durability setting. You can fire-and-forget with `acks=all` (the broker is very careful) just as easily as with `acks=0` (the broker answers immediately). `acks` and "do I look at the result" are independent choices. |
| Some `mode="async"` flag | No such flag exists in `confluent-kafka`. Every `produce()` call is asynchronous (Notebook 03, Step 2) — fire-and-forget or not. |
| A different producer class | Same `Producer` class for every pattern in this folder. Patterns 1–5 differ only in *how the surrounding code is written*, never in which client object is used. |

**So when you read Section 1 below, watch for the missing `on_delivery=`.** That's the whole mechanism.

## Section 1 — The minimum fire-and-forget send

Here is the whole pattern. Four lines of real work. Read each comment.

In [3]:
recreate_topic("ff_demo")

# Stage 1 - Create the producer. The config dict needs only the broker address.
#           We do NOT set a delivery callback. That omission is the whole point:
#           "fire and forget" means we are choosing not to look at the result.
producer = Producer({"bootstrap.servers": BOOTSTRAP})

# Stage 2 - Build the message body. Kafka stores bytes, so we turn our Python
#           dictionary into a JSON string, then into UTF-8 bytes.
event = {"user": "alice", "action": "page_view", "page": "/home"}
value_bytes = json.dumps(event).encode("utf-8")

# Stage 3 - produce() does NOT send anything over the network. It drops the message
#           into an in-memory queue inside the producer and returns instantly.
producer.produce("ff_demo", value=value_bytes)
print("produce() returned. The message is queued in memory, not yet on the broker.")

# Stage 4 - flush() blocks until every queued message has actually been sent and the
#           broker has answered (or it failed). We will remove this line in Section 3
#           to show what breaks without it.
producer.flush()
print("flush() returned. Now the message is really on the broker.")

Topic 'ff_demo' ready with 3 partitions.
produce() returned. The message is queued in memory, not yet on the broker.
flush() returned. Now the message is really on the broker.


Confirm it landed. The sum of the three partitions' end offsets should be 1:

In [4]:
kafka_tool("kafka-get-offsets.sh", "--bootstrap-server", "localhost:9092", "--topic", "ff_demo", "--time", "latest")

ff_demo:0:0
ff_demo:1:1
ff_demo:2:0


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-get-offsets.sh', '--bootstrap-server', 'localhost:9092', '--topic', 'ff_demo', '--time', 'latest'], returncode=0, stdout='ff_demo:0:0\nff_demo:1:1\nff_demo:2:0\n', stderr='')


# Section 1.5 — Understanding Kafka Producer: `produce()`, `poll()`, and `flush()`

## 1. The Courier Company Analogy

Imagine you own an online shopping business.

Every day, your business sends hundreds of parcels to customers through a courier company.

You have a courier desk in your office where you hand over parcels. The courier company takes responsibility for transporting them and reporting whether delivery succeeded or failed.

This is very similar to how a Kafka Producer works.

Let's understand each operation step by step.

---

## Step 1 — `produce()`: Hand Over a Parcel

You prepare a parcel containing an order and place it on the courier desk.

The courier employee accepts it into the outgoing parcel queue.

You immediately return to your work.

**Important:** The parcel has not necessarily reached its destination. It has merely been accepted for delivery.

### In Kafka

```python
producer.produce(
    topic="orders",
    value="Order 101"
)
```

**What happens?**

1. Your application calls `produce()`.
2. The Kafka Producer accepts the message into its internal queue.
3. The method normally returns immediately without waiting for delivery.
4. The message is now waiting to be transmitted, or may already be in transit.

**Remember:**

`produce()` means **"Please send this message."**

It does not mean **"The message has successfully reached Kafka."**

---

## Step 2 — Background Networking: The Courier Transports the Parcel

After you hand over the parcel, the courier company automatically collects it and transports it to the destination.

You do not have to call the courier employee repeatedly.

You do not have to tell the driver when to start.

The courier company handles transportation independently.

### In Kafka

When you create a Producer:

```python
producer = Producer({
    "bootstrap.servers": "localhost:9092"
})
```

The `confluent-kafka` library uses an underlying library called **librdkafka**.

This library manages background networking threads responsible for:

- Connecting to Kafka brokers.
- Collecting queued messages into batches.
- Sending messages over the network.
- Receiving broker responses.
- Handling retries when appropriate.

**Important distinction:**

`produce()` queues the message.

**The producer's background networking actually sends the message to Kafka.**

You do not need to call `poll()` or `flush()` to start this transmission.

---

## Step 3 — Broker Acknowledgement: The Destination Signs the Receipt

Imagine the courier successfully delivers your parcel.

The recipient signs a delivery receipt.

The courier company receives confirmation that the parcel was delivered.

Now the courier company knows the delivery result.

However, you have not necessarily checked that result yet.

### In Kafka

The background networking sends your message to the Kafka broker.

The broker processes the message and, depending on the configured acknowledgement policy, sends a response.

Kafka supports different acknowledgement settings:

| Setting | Meaning |
|---|---|
| `acks=0` | The producer does not wait for broker acknowledgement. |
| `acks=1` | The leader broker acknowledges the message. |
| `acks=all` | The broker acknowledges after the required in-sync replicas have accepted the message. |

**Important:**

A broker acknowledgement confirms delivery according to the configured acknowledgement policy.

It does **not** mean that a Kafka Consumer has already read the message.

For `acks=0`, there is no broker acknowledgement to confirm receipt.

---

## Step 4 — `producer.poll()`: Process Delivery Notifications

Imagine your courier company maintains a notification inbox.

Whenever a parcel reaches its final delivery outcome, a delivery report becomes available.

For example:

- Parcel 101 — Delivered successfully.
- Parcel 102 — Delivered successfully.
- Parcel 103 — Delivery failed.

These reports are waiting to be processed.

You open the notification inbox and process the available reports.

### In Kafka

When the producer receives a final delivery result, it makes a delivery event available for processing.

Your Python callback normally runs when you call:

```python
producer.poll(0)
```

**What does `poll(0)` mean?**

Process producer events that are already available without waiting for additional events.

If a delivery report is ready, the corresponding callback can execute.

### Example

```python
def delivery_report(error, message):

    if error is not None:
        print("Delivery failed:", error)

    else:
        print("Message delivered successfully!")


producer.produce(
    topic="orders",
    value="Order 101",
    on_delivery=delivery_report
)

producer.poll(0)
```

**Important:**

`producer.poll()` does not send the message.

It does not contact the broker to ask whether that particular message was delivered.

The producer's background networking handles sending and receiving delivery results.

`producer.poll()` processes available events and executes callbacks.

### What about the timeout?

| Method | Meaning |
|---|---|
| `producer.poll(0)` | Process available events without waiting. |
| `producer.poll(1)` | Wait up to 1 second while serving events. |
| `producer.poll(2)` | Wait up to 2 seconds while serving events. |

**Remember:**

`producer.poll()` means **"Process any delivery notifications that are ready."**

It may also process other producer events, such as error or statistics callbacks, when configured.

---

## Step 5 — `producer.flush()`: Wait Before Closing the Office

Imagine it is closing time at your business.

You handed over 100 parcels today.

Some have been delivered.

Others might still be in transit.

Before shutting down your office, you want every outstanding parcel to reach a final delivery outcome.

You tell the courier company:

**"Wait until every parcel has a delivery result, or until our waiting deadline expires."**

That is what `flush()` does.

### In Kafka

```python
producer.flush()
```

This waits for outstanding producer messages to complete their delivery attempts.

While waiting, `flush()` also processes producer events and executes delivery callbacks.

Conceptually, it behaves like repeatedly calling `poll()` until the producer has no outstanding messages or the timeout expires.

### Example

```python
producer.produce(
    topic="orders",
    value="Order 101",
    on_delivery=delivery_report
)

remaining = producer.flush(10)
```

Here:

1. `produce()` queues the message.
2. Background networking sends it automatically.
3. `flush(10)` waits up to 10 seconds for outstanding deliveries to complete.
4. Delivery callbacks are processed while waiting.
5. The return value indicates how many messages remain outstanding.

**Important:**

`flush()` does not initiate message transmission.

It waits for the producer's outstanding work to finish.

Also, a successful `flush()` does not guarantee every message was delivered successfully. Some deliveries may have failed.

You must inspect delivery callbacks to determine success or failure.

---

## 2. Complete Kafka Producer Workflow

```text
YOUR PYTHON APPLICATION
        |
        |
        | producer.produce("orders", "Order 101")
        |
        v
+---------------------------+
| Producer's Internal Queue |
|                           |
| Order 101                 |
+---------------------------+
        |
        | Automatically handled
        v
+---------------------------+
| Background Networking     |
|                           |
| Sends messages to Kafka   |
+---------------------------+
        |
        | Network transmission
        v
+---------------------------+
| Kafka Broker              |
|                           |
| Receives the message      |
| Sends acknowledgement     |
| according to acks policy  |
+---------------------------+
        |
        | Response received by
        | background networking
        v
+---------------------------+
| Producer Delivery Event   |
|                           |
| Success or failure        |
+---------------------------+
        |
        | producer.poll()
        | OR producer.flush()
        v
+---------------------------+
| Your Python Callback      |
|                           |
| delivery_report(...)      |
+---------------------------+
```

**Note:** This diagram represents the logical message lifecycle. Background networking runs independently, and you do not have to call `poll()` before `flush()`.

---

## 3. Mapping the Analogy to Kafka

| Courier Company | Kafka Producer |
|---|---|
| Hand over a parcel | `produce()` |
| Outgoing parcel queue | Producer's internal queue |
| Courier transportation system | Background networking |
| Destination signs receipt | Broker acknowledgement |
| Delivery status report | Delivery report |
| Process notification inbox | `producer.poll()` |
| Wait for pending parcels before closing | `producer.flush()` |

---

## 4. Producer `poll()` vs Consumer `poll()`

**This is one of the most important distinctions in Kafka.**

Both the Kafka Producer and Kafka Consumer have a method named `poll()`.

However, they perform different operations.

### Producer

```python
producer.poll(0)
```

**Purpose:** Process producer events, including delivery reports, and execute registered callbacks.

It does not consume messages from Kafka topics.

### Consumer

```python
message = consumer.poll(1.0)
```

**Purpose:** Retrieve an available Kafka message or consumer event.

This is how your application receives messages for processing.

### Comparison

| Method | Responsibility |
|---|---|
| `producer.produce()` | Queue a message for transmission. |
| `producer.poll()` | Process producer events and delivery callbacks. |
| `producer.flush()` | Wait for outstanding deliveries to finish. |
| `consumer.poll()` | Retrieve messages or events for the consumer. |

---

## 5. Do We Always Need `producer.poll()`?

**No.**

For a small script, this is perfectly valid:

```python
producer.produce(
    topic="orders",
    value="Order 101",
    on_delivery=delivery_report
)

producer.flush()
```

Why?

Because `flush()` internally processes producer events, including delivery callbacks.

You do not need to call `producer.poll()` separately in this example.

However, in long-running applications, calling `poll(0)` periodically helps process delivery reports without waiting for every message to finish.

### Common Producer Pattern

```python
for i in range(1000):

    producer.produce(
        topic="orders",
        value=f"Order {i}",
        on_delivery=delivery_report
    )

    # Process ready delivery reports.
    producer.poll(0)


# Before shutting down, wait for outstanding messages.
producer.flush()
```

This pattern allows asynchronous message publishing.

**Production note:** You should additionally handle a full producer queue, delivery failures, and flush timeouts.

---

## 6. The Final Mental Model

Think about sending 100 parcels.

**`produce()`**

You hand over 100 parcels to the courier company.

You do not personally transport them.

**Background Networking**

The courier company automatically transports the parcels.

This is what actually sends the Kafka messages.

**Broker Acknowledgement**

The destination confirms receipt according to the configured acknowledgement policy.

**`producer.poll()`**

You process available delivery notifications.

Your Python callback functions execute.

**`producer.flush()`**

You wait for all outstanding parcels to reach a final outcome before closing, or until your timeout expires.

**`consumer.poll()`**

A completely separate operation: the recipient collects incoming parcels.

---

## 7. One-Sentence Summary

> **`produce()` queues the message, background networking sends it, the broker acknowledges it, `producer.poll()` processes delivery notifications, and `producer.flush()` waits for outstanding deliveries to finish.**

### Memory Trick

**QUEUE → SEND → ACKNOWLEDGE → REPORT → FINISH**

| Action | Responsible component |
|---|---|
| QUEUE | `produce()` |
| SEND | Background networking |
| ACKNOWLEDGE | Kafka broker |
| REPORT | `producer.poll()` or `producer.flush()` |
| FINISH | `producer.flush()` |

Once you understand these responsibilities, Kafka producer patterns become much easier to follow.


In [4]:

"""
KAFKA PRODUCER — UNDERSTANDING MESSAGE DELIVERY
================================================

Scenario:
    Our e-commerce application publishes customer orders
    to Kafka.

What we want to learn:
    1. produce() queues messages for sending.
    2. Background networking sends them automatically.
    3. poll() executes delivery callbacks.
    4. flush() waits for outstanding deliveries.

We also follow useful production practices:
    - JSON serialization
    - Message keys
    - Delivery callbacks
    - Error handling
    - Bounded shutdown
"""

import json
import time

from confluent_kafka import Producer


# ============================================================
# STEP 1: CONFIGURATION
# ============================================================

TOPIC = "orders_demo"
TOTAL_ORDERS = 20

# Start with a fresh topic for our learning experiment.
# WARNING: This deletes existing messages in this topic.
# Never do this automatically in production.
recreate_topic(TOPIC, partitions=3)


# ============================================================
# STEP 2: TRACK MESSAGE DELIVERY
# ============================================================

# These counters help us understand what Kafka is doing.
#
# queued:
#   Number of messages accepted by produce().
#
# delivered:
#   Number of successful delivery callbacks.
#
# failed:
#   Number of failed delivery callbacks.

stats = {
    "queued": 0,
    "delivered": 0,
    "failed": 0
}


# ============================================================
# STEP 3: DEFINE THE DELIVERY CALLBACK
# ============================================================

def on_delivery(error, message):
    """
    Called when Kafka reports the final delivery outcome.

    IMPORTANT:
    This function normally executes when our Python code
    calls producer.poll() or producer.flush().

    It does NOT send the message or request acknowledgement.
    Background networking already handles those operations.
    """

    if error is not None:

        stats["failed"] += 1

        print(
            f"FAILED | key={message.key()} | "
            f"reason={error}"
        )

    else:

        stats["delivered"] += 1

        print(
            f"DELIVERED | "
            f"partition={message.partition()} | "
            f"offset={message.offset()}"
        )


# ============================================================
# STEP 4: CREATE THE PRODUCER
# ============================================================

producer = Producer({
    "bootstrap.servers": BOOTSTRAP,

    # Wait for the strongest available broker acknowledgement
    # under the topic's replication configuration.
    "acks": "all",

    # Prevent duplicate writes caused by eligible retries.
    "enable.idempotence": True,

    # Limit how long delivery may take before failing.
    "delivery.timeout.ms": 30000,
})


# ============================================================
# STEP 5: PREPARE A CUSTOMER ORDER
# ============================================================

def create_order(order_id):
    """
    Create one example order.

    In a real application, this data might come from
    an API request, database, or another service.
    """

    return {
        "order_id": order_id,
        "customer_id": 1001,
        "product": "Laptop",
        "quantity": 1,
        "price": 80000
    }


# ============================================================
# STEP 6: PUBLISH ONE ORDER
# ============================================================

def publish_order(order):
    """
    Queue an order for asynchronous delivery to Kafka.

    This function does not wait for broker acknowledgement.
    """

    # Kafka messages contain bytes.
    # Convert our Python dictionary into JSON bytes.
    message_value = json.dumps(order).encode("utf-8")

    # Use the order ID as the message key.
    # The key helps Kafka choose a partition consistently.
    message_key = str(order["order_id"]).encode("utf-8")

    while True:
        try:

            producer.produce(
                topic=TOPIC,
                key=message_key,
                value=message_value,
                on_delivery=on_delivery
            )

            # The producer accepted the message locally.
            # This does NOT mean Kafka delivered it.
            stats["queued"] += 1

            return

        except BufferError:

            # The local producer queue is full.
            # Process events and allow room to become available.
            print("Producer queue full. Waiting for space...")

            producer.poll(1.0)


# ============================================================
# STEP 7: STAGE A — QUEUE ORDERS
# ============================================================

print("\nSTAGE A: Queueing customer orders")
print("-" * 50)

for order_id in range(1, TOTAL_ORDERS + 1):

    order = create_order(order_id)

    publish_order(order)

    # Deliberately no poll() here.
    #
    # This is ONLY for our learning experiment.
    # Background networking can still send messages.


print(f"Orders accepted locally: {stats['queued']}")
print(f"Callbacks processed: {stats['delivered'] + stats['failed']}")
print(f"Outstanding producer queue: {len(producer)}")


# ============================================================
# STEP 8: STAGE B — ALLOW BACKGROUND NETWORKING TO WORK
# ============================================================

print("\nSTAGE B: Waiting without calling poll()")
print("-" * 50)

# Our Python thread sleeps for 2 seconds.
#
# Kafka's background networking continues independently.
# Messages may be sent and acknowledged during this time.
#
# But delivery callbacks normally remain unprocessed
# until we call poll() or flush().

time.sleep(2)

print("Python finished sleeping.")

print(f"Callbacks processed: {stats['delivered'] + stats['failed']}")
print(f"Outstanding producer queue: {len(producer)}")

# NOTE:
# len(producer) counts outstanding messages AND queued events.
# It is NOT simply the number of undelivered messages.


# ============================================================
# STEP 9: STAGE C — PROCESS DELIVERY REPORTS
# ============================================================

print("\nSTAGE C: Processing delivery reports")
print("-" * 50)

# Process events that are already available.
# Do not wait for new events.

producer.poll(0)

print(f"Successfully delivered: {stats['delivered']}")
print(f"Failed deliveries: {stats['failed']}")
print(f"Outstanding producer queue: {len(producer)}")

# IMPORTANT:
# poll() did not cause Kafka to send the messages.
#
# It allowed Python to execute delivery callbacks
# for events that were already ready.


# ============================================================
# STEP 10: STAGE D — GRACEFUL SHUTDOWN
# ============================================================

print("\nSTAGE D: Waiting for remaining deliveries")
print("-" * 50)

# Wait up to 35 seconds for outstanding messages.
#
# flush() also processes delivery callbacks internally.
#
# Its return value is the number of messages/events
# still outstanding when it returns.

remaining = producer.flush(timeout=35)

if remaining == 0:
    print("All outstanding producer work completed.")
else:
    print(f"WARNING: {remaining} messages/events remain.")


# ============================================================
# STEP 11: FINAL DELIVERY SUMMARY
# ============================================================

print("\nFINAL SUMMARY")
print("=" * 50)

print(f"Orders queued:       {stats['queued']}")
print(f"Orders delivered:    {stats['delivered']}")
print(f"Orders failed:       {stats['failed']}")
print(f"Still outstanding:   {remaining}")

if remaining == 0 and stats["failed"] == 0:
    print("\nSUCCESS: All queued orders were delivered.")
else:
    print("\nWARNING: Some orders failed or remain outstanding.")


Topic 'orders_demo' ready with 3 partitions.

STAGE A: Queueing customer orders
--------------------------------------------------
Orders accepted locally: 20
Callbacks processed: 0
Outstanding producer queue: 20

STAGE B: Waiting without calling poll()
--------------------------------------------------
Python finished sleeping.
Callbacks processed: 0
Outstanding producer queue: 23

STAGE C: Processing delivery reports
--------------------------------------------------
DELIVERED | partition=0 | offset=0
DELIVERED | partition=0 | offset=1
DELIVERED | partition=0 | offset=2
DELIVERED | partition=0 | offset=3
DELIVERED | partition=0 | offset=4
DELIVERED | partition=0 | offset=5
DELIVERED | partition=0 | offset=6
DELIVERED | partition=0 | offset=7
DELIVERED | partition=0 | offset=8
DELIVERED | partition=1 | offset=0
DELIVERED | partition=1 | offset=1
DELIVERED | partition=1 | offset=2
DELIVERED | partition=1 | offset=3
DELIVERED | partition=1 | offset=4
DELIVERED | partition=1 | offset=5
D

**The jargon, defined once:**

| Word | Plain meaning |
|---|---|
| **Fire-and-forget** | Send and do not check the outcome. |
| **Asynchronous** | `produce()` returns *before* the work is done; the work happens later on a background thread. |
| **Delivery callback** | A function Kafka would call to tell you "delivered" or "failed". Fire-and-forget *skips* this. |
| **`flush()`** | "Wait here until everything queued is actually sent." |

## Section 2 — What breaks if you NEVER poll at all

Section 1.5 already showed what `poll()` does: process whatever delivery reports have arrived. Here's the concrete *failure* you get if you skip it entirely, forever, while producing quickly.

The producer's local queue (the one `produce()` adds to) has a size limit, set by `queue.buffering.max.messages` (default 100,000). If you never call `poll()`, finished reports never get cleared out of it -- and new messages can't be accepted once it's full. `produce()` then raises `BufferError`, Kafka's way of saying "I can't take any more until you process what's already waiting."

**You might reasonably object:** "But surely, if I wait a while, the broker finishes delivering a bunch of these on its own, and *that* frees up room -- even without `poll()`?" That's a fair question, and we test it directly below instead of just asserting the answer.

In [5]:
# ============================================================
# STEP 1: EXPERIMENT SETTINGS
# ============================================================

# We use a small limit on purpose, so BufferError shows up fast
# instead of only after 100,000 messages (the real-world default).
QUEUE_SIZE_LIMIT = 2000

# How many messages we will TRY to send. More than the limit, so we
# are guaranteed to hit BufferError before running out of messages.
MESSAGES_TO_ATTEMPT = 5000

# We pause briefly every this-many messages, to give the broker real
# wall-clock time to deliver and acknowledge what we've already sent.
MESSAGES_PER_PAUSE = 200
PAUSE_SECONDS = 0.3


# ============================================================
# STEP 2: CREATE A PRODUCER WITH A SMALL QUEUE LIMIT
# ============================================================

recreate_topic("ff_poll")
producer = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "queue.buffering.max.messages": QUEUE_SIZE_LIMIT,
})


# ============================================================
# STEP 3: DEFINE THE EXPERIMENT AS ONE FUNCTION
# ============================================================

def produce_without_ever_polling(producer, topic, max_messages):
    """Produce messages in small bursts, pausing for real time between
    them, but NEVER call poll() -- not even once.

    Why this exists: this is the whole experiment. We want to know
    whether skipping poll() forever eventually breaks produce() itself,
    even when the broker is healthy and has plenty of real time to
    deliver and acknowledge every message we send it.

    Returns:
        A dict with:
        - messages_sent: how many produce() calls succeeded before failing
        - elapsed_seconds: how much REAL wall-clock time passed
        - error_message: the BufferError's message, or None if we never hit it
    """
    messages_sent = 0
    start_time = time.time()

    try:
        for message_number in range(max_messages):
            producer.produce(topic, value=b"x")
            messages_sent += 1

            # Every MESSAGES_PER_PAUSE messages, stop and sleep for real
            # time. This gives the broker a genuine chance to finish
            # delivering and acknowledging earlier messages on its own --
            # while we still call NO poll() at all, on purpose.
            just_finished_a_batch = message_number % MESSAGES_PER_PAUSE == 0
            if just_finished_a_batch:
                time.sleep(PAUSE_SECONDS)

    except BufferError as error:
        elapsed_seconds = time.time() - start_time
        return {
            "messages_sent": messages_sent,
            "elapsed_seconds": elapsed_seconds,
            "error_message": str(error),
        }

    # If we get here, we never hit BufferError within max_messages.
    elapsed_seconds = time.time() - start_time
    return {
        "messages_sent": messages_sent,
        "elapsed_seconds": elapsed_seconds,
        "error_message": None,
    }


# ============================================================
# STEP 4: RUN THE EXPERIMENT
# ============================================================

result = produce_without_ever_polling(producer, "ff_poll", MESSAGES_TO_ATTEMPT)

if result["error_message"] is not None:
    print(f"BufferError after sending {result['messages_sent']} messages,")
    print(f"after {result['elapsed_seconds']:.1f} REAL seconds, with ZERO poll() calls.")
    print(f"error: {result['error_message']}")
else:
    print(f"No BufferError. All {result['messages_sent']} messages were accepted.")


# ============================================================
# STEP 5: DID THE BROKER ALREADY FINISH MOST DELIVERIES ANYWAY?
# ============================================================

def count_pending_before_and_after_one_poll(producer):
    """Check how many messages have 'no outcome processed yet' right now,
    then call poll() exactly once, and check again.

    Why this exists: this is how we PROVE whether completed deliveries
    were silently freeing up queue room on their own (they were not --
    see the explanation below).
    """
    pending_before = len(producer)
    producer.poll(0)      # process whatever delivery reports are ready
    pending_after = len(producer)
    return pending_before, pending_after


pending_before_poll, pending_after_poll = count_pending_before_and_after_one_poll(producer)

print()
print(f"Pending (no outcome processed) right when BufferError hit: {pending_before_poll}")
print(f"Pending after a SINGLE poll(0) call:                       {pending_after_poll}")

producer.flush()

Topic 'ff_poll' ready with 3 partitions.
BufferError after sending 2000 messages,
after 3.0 REAL seconds, with ZERO poll() calls.
error: Local: Queue full

Pending (no outcome processed) right when BufferError hit: 2010
Pending after a SINGLE poll(0) call:                       199


0

**Read the two pending numbers carefully.** `BufferError` hit right around our 2000-message limit, *despite* over 3 real seconds having passed with zero `poll()` calls -- plenty of time for a healthy localhost broker to deliver and acknowledge thousands of tiny messages. Yet when we finally called `poll(0)` **exactly once**, the pending count dropped by well over a thousand almost instantly.

That drop is the proof: those messages had already been delivered and acknowledged by the broker long before the error -- they were just sitting there, completed, with nobody having processed their delivery report. The queue slot they occupied was never released until `poll()` ran.

**The conclusion, stated precisely:** a completed delivery does not free its queue slot on its own. The slot is only released when `poll()` (or `flush()`) processes that message's delivery report -- success or failure, either one (Section 1.5's point). "Background networking delivers it" and "the local queue has room for a new message" are two separate facts; only calling `poll()` connects them.

**This is called back-pressure**: Kafka's way of forcing your code to slow down and deal with results instead of piling up forever. The standard way to handle it:

```python
try:
    producer.produce(topic, value=value)
except BufferError:
    producer.poll(0.5)        # make room by processing what's ready, waiting a little if needed
    producer.produce(topic, value=value)   # now retry
```

> **The standard fire-and-forget loop is: `produce()` then `poll(0)`, repeated; `flush()` once at the very end.** `poll(0)` is cheap and keeps the queue from ever reaching this point. `flush()` at the end does a final, blocking drain.

## Section 3 — Why `flush()` at the end is not optional

`produce()` only *queues*. If your program ends while messages are still in the outgoing queue, **those messages are gone** — they never reached Kafka. `flush()` is the instruction "wait until the queue is empty before continuing".

We simulate a program that forgets to flush by sending, then immediately deleting the producer (which is what happens when a short script exits).

In [6]:
# ============================================================
# STEP 1: EXPERIMENT SETTINGS
# ============================================================

MESSAGES_TO_SEND = 1000


# ============================================================
# STEP 2: DEFINE THE EXPERIMENT AS ONE FUNCTION
# ============================================================

def send_then_abandon_without_flushing(topic, message_count):
    """Send messages, then throw the producer away WITHOUT calling flush().

    Why this exists: this simulates a real mistake -- a short script, or a
    crashing program, that ends before giving the producer a chance to
    finish sending everything that was queued.

    Returns:
        How many messages were still waiting (no outcome processed) at the
        exact moment we abandoned the producer.
    """
    producer = Producer({"bootstrap.servers": BOOTSTRAP})

    for message_number in range(message_count):
        producer.produce(topic, value=b"x")
        producer.poll(0)      # process whatever is ready so far -- but see Step 4 for why this is NOT enough

    messages_still_pending = len(producer)

    # del producer simulates the program ending here: no flush() is called,
    # so anything still pending never gets a chance to finish sending.
    del producer

    return messages_still_pending


# ============================================================
# STEP 3: RUN THE EXPERIMENT
# ============================================================

recreate_topic("ff_flush")
pending_when_abandoned = send_then_abandon_without_flushing("ff_flush", MESSAGES_TO_SEND)
print(f"Messages still pending when we abandoned the producer (no flush): {pending_when_abandoned}")


# ============================================================
# STEP 4: CHECK WHAT THE BROKER ACTUALLY RECEIVED
# ============================================================

# Give the broker a moment to settle, then ask IT (not our Python code)
# how many messages each partition actually ended up holding.
time.sleep(1)
print("\nWhat the broker actually stored (end offset per partition):")
kafka_tool("kafka-get-offsets.sh", "--bootstrap-server", "localhost:9092", "--topic", "ff_flush", "--time", "latest")

Topic 'ff_flush' ready with 3 partitions.
Messages still pending when we abandoned the producer (no flush): 1000


%4|1792149186.322|TERMINATE|rdkafka#producer-5| [thrd:app]: Producer terminating with 1000 messages (1000 bytes) still in queue or transit: use flush() to wait for outstanding message delivery



What the broker actually stored (end offset per partition):
ff_flush:0:0
ff_flush:1:0
ff_flush:2:0


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-get-offsets.sh', '--bootstrap-server', 'localhost:9092', '--topic', 'ff_flush', '--time', 'latest'], returncode=0, stdout='ff_flush:0:0\nff_flush:1:0\nff_flush:2:0\n', stderr='')

The end offsets add up to **fewer than 1000** (often far fewer): the messages that were still queued when we dropped the producer never made it. In a real service this is data silently thrown away on shutdown.

**Rule:** always `flush()` before your program exits. The repo's version uses Python's `atexit` to call `flush()` automatically — a nice touch, but the principle is simply "flush before you go".

**One more thing you'll see in the real output, that we didn't print ourselves:** a line like

```text
%4|...|TERMINATE|rdkafka#producer-N| [thrd:app]: Producer terminating with 1000 messages (1000 bytes) still in queue or transit: use flush() to wait for outstanding message delivery
```

That's **librdkafka itself** (the C library underneath `confluent-kafka`) printing a warning automatically, the moment it notices it's being destroyed (our `del producer`) while messages are still unsent. It is not an error from our code -- it's the library independently confirming the exact lesson of this cell: we abandoned it without flushing, and it's telling us so. If you ever see a `TERMINATE` warning like this in a real application's logs, it means exactly this happened for real.


## Section 4 — The defining weakness: silent failure

Fire-and-forget's danger is not just crash-on-exit. Even while running perfectly, **if a message fails to deliver, you are never told**, because you registered no callback.

We prove it by pointing a producer at a dead address (port 9999, where nothing listens) and sending. With no callback, the code looks completely successful.

In [7]:
# A producer aimed at nothing. message.timeout.ms=3000 = "give up after 3 seconds".
doomed = Producer({"bootstrap.servers": "localhost:9999", "message.timeout.ms": 3000, "log_level": 0})

doomed.produce("ff_demo", value=b"this message will die quietly")
doomed.poll(0)
print("Our code sees: nothing wrong. produce() and poll() did not complain.")

doomed.flush()        # waits out the 3s timeout
print("After flush, our code STILL sees nothing. The message was lost with no error raised to us.")

Our code sees: nothing wrong. produce() and poll() did not complain.
After flush, our code STILL sees nothing. The message was lost with no error raised to us.


Nothing in our code learned about the failure. That is fire-and-forget by definition. 

**Contrast preview:** the *next* pattern, **callback-confirmed**, adds exactly one thing — a callback — and suddenly that same lost message produces a visible error. That single difference is the entire reason the next pattern exists.

## Section 5 — The payoff: speed

Why accept silent loss at all? Because not waiting is **fast**. We send 50,000 messages fire-and-forget and measure throughput.

In [8]:
recreate_topic("ff_speed", partitions=3)
producer = Producer({"bootstrap.servers": BOOTSTRAP})

count = 50_000
start = time.time()
for i in range(count):
    producer.produce("ff_speed", value=json.dumps({"n": i}).encode("utf-8"))
    producer.poll(0)
producer.flush()
elapsed = time.time() - start

print(f"Sent {count:,} messages in {elapsed:.2f} s  =  {count/elapsed:,.0f} messages/second")

Topic 'ff_speed' ready with 3 partitions.
Sent 50,000 messages in 0.42 s  =  119,414 messages/second


Tens of thousands of messages per second from a single thread on a laptop, because we never pause to wait for the broker. That throughput is the whole reason this pattern exists. Use it where **volume matters more than any single message**.

## Section 6 — Wrap it in a tiny reusable class

So far we called `produce`/`poll`/`flush` directly. Real apps wrap this in a small object so the rest of the code just calls `.send(...)`. Here is the **entire** useful pattern in ~20 lines. Compare this to the repo's 800-line file — everything essential is here.

In [ ]:
class FireAndForgetProducer:
    """A minimal fire-and-forget producer.

    Why it exists: give the rest of the app a dead-simple .send() that never blocks,
    while correctly draining delivery reports (poll) and flushing on close.

    What it does NOT do (on purpose): it does not tell you if a message failed.
    If you need that, use the callback-confirmed pattern (next notebook).
    """

    def __init__(self, bootstrap_servers: str):
        # One producer, reused for every send. (Section 8 and the next notebook
        # explain why one shared instance beats creating a new producer per message.)
        self._producer = Producer({"bootstrap.servers": bootstrap_servers})

    def send(self, topic: str, data: dict, key: str | None = None) -> None:
        """Queue one message and return immediately. No delivery guarantee."""
        value = json.dumps(data).encode("utf-8")
        key_bytes = key.encode("utf-8") if key else None
        # No on_delivery= here. That omission is the ENTIRE fire-and-forget mechanism.
        self._producer.produce(topic, value=value, key=key_bytes)
        self._producer.poll(0)        # service any delivery results that are ready

    def close(self) -> None:
        """Flush remaining messages. Always call before the program exits."""
        self._producer.flush()


# Use it:
recreate_topic("ff_class")
sender = FireAndForgetProducer(BOOTSTRAP)
for i in range(10):
    sender.send("ff_class", {"event": "click", "n": i}, key="user-1")
sender.close()

print("Sent 10 via the class. End offsets:")
kafka_tool("kafka-get-offsets.sh", "--bootstrap-server", "localhost:9092", "--topic", "ff_class", "--time", "latest")

That class **is** the fire-and-forget pattern. Everything beyond it in the repo (health counters, singleton machinery, optional confirmation, retries) belongs to *other* patterns that happen to be bolted onto the same file — see Section 8.

## Section 7 — When to use it (and when never)

| Use fire-and-forget when… | Do NOT use it when… |
|---|---|
| High volume, each message cheap | Each message is valuable (payments, orders) |
| Losing a few is acceptable | Loss must be detected or prevented |
| Examples: metrics, logs, clickstream, telemetry, IoT sensor pings | Examples: financial transactions, user sign-ups, inventory changes |
| You care about throughput above all | You need to confirm or react to delivery |

**Decision rule:** *"If one of these messages vanished and nobody noticed, is that OK?"* If yes → fire-and-forget. If no → you need the callback-confirmed pattern.

**Can you make it safer without leaving the pattern?** Partly. Set `acks=all`, `enable.idempotence=True`, and a generous `message.timeout.ms`, so the *broker side* is durable and retried. But you still won't be *notified* of a final failure — that notification is what the next pattern adds.

## Section 8 — What the repo's 800-line version actually contains

Open `producers/fire_and_forget/core.py`. It is ~800 lines. You asked why it feels like being hit by a truck — here is the honest breakdown. Most of it is **not fire-and-forget**; it is four other concerns stacked on top:

| Lines of that file do… | Which is really… | Taught in |
|---|---|---|
| `produce()` + `poll(0)` + return | **actual fire-and-forget** (what Section 6 shows) | this notebook |
| `__new__`, `get_instance`, `reset_instance` locks | **Singleton pattern** (one shared producer) | Notebook 3 (singleton) |
| health counters, error-rate guard, `_is_healthy` | **Health monitoring** (an operations concern) | Notebook 5 (resilient) |
| `require_delivery_confirmation`, `_await_delivery_confirmation`, `threading.Event` | **Callback-confirmed pattern** (the opposite of fire-and-forget!) | Notebook 2 (callbacks) |
| `send_with_retry`, backoff loops | **Retry pattern** | Notebook 5 (resilient) |
| `sanitize_config_for_logging`, input validation | good hygiene, but noise while learning | — |

So the file is confusing because it answers five questions at once. We are answering them **one notebook at a time**. When you finish all five, re-read that file and it will look like five small ideas you already know, not one monster.

## Section 9 — Clean up and checkpoint

In [7]:
for t in ["ff_demo", "orders_demo", "ff_poll", "ff_flush", "ff_speed", "ff_class"]:
    if t in admin.list_topics(timeout=10).topics:
        admin.delete_topics([t])[t].result()
print("Cleaned up. Remaining non-internal topics:",
      sorted(t for t in admin.list_topics(timeout=10).topics if not t.startswith("__")))

Cleaned up. Remaining non-internal topics: ['ff_flush', 'nb3_acks', 'orders']


%5|1792149601.356|PARTCNT|rdkafka#producer-4| [thrd:main]: Topic ff_poll (id iKUkw+VFRKKpxNFbOzPvtA) partition count changed from 3 to 0



# Kafka Fire-and-Forget Producer — What Engineers Actually Remember

## 1. What is Fire-and-Forget?

Fire-and-forget means the application **publishes messages asynchronously without tracking individual delivery outcomes**.

The application submits messages and continues its work without waiting for each broker acknowledgement.

**Typical use cases:**
- Non-critical logs
- Disposable telemetry
- Low-priority analytics events

**Avoid pure fire-and-forget for:** Payments, critical orders, inventory updates, or events where losing a message has serious consequences.

> **Remember:** Fire-and-forget is about not tracking delivery results. It does not necessarily mean Kafka is configured with `acks=0`.

---

## 2. The Only Three Producer Methods to Remember

| Method | What it actually does | When engineers use it |
|---|---|---|
| `produce()` | Queues a message for asynchronous sending | Every time an event is published |
| `producer.poll(0)` | Processes ready producer events and callbacks without blocking | Regularly during publishing |
| `flush(timeout)` | Waits for outstanding deliveries to finish or timeout | Before shutdown or at a batch boundary |

**Who actually sends the message?**

The producer's background networking (`librdkafka`), not `poll()` or `flush()`.

### Mental Model

```text
Application
    |
    | produce()
    v
Local Producer Queue
    |
    | Background networking sends automatically
    v
Kafka Broker
    |
    | Acknowledgement according to acks policy
    v
Producer receives delivery result
    |
    | poll() / flush() processes events
    v
Application callback (if registered)
```

---

## 3. Standard Fire-and-Forget Pattern

```python
import json
from confluent_kafka import Producer

producer = Producer({
    "bootstrap.servers": "localhost:9092"
})

try:
    for order_id in range(100):

        event = {
            "order_id": order_id,
            "event": "order_created"
        }

        # Queue the message; do not wait for delivery.
        producer.produce(
            topic="events",
            key=str(order_id),
            value=json.dumps(event).encode("utf-8")
        )

        # Process ready producer events.
        producer.poll(0)

finally:
    # Wait for outstanding work before shutdown.
    remaining = producer.flush(timeout=30)

    if remaining:
        print(f"WARNING: {remaining} messages/events outstanding")
```

**Why this pattern?**

1. Create one reusable producer.
2. Use `produce()` to submit messages asynchronously.
3. Use `poll(0)` regularly to service producer events.
4. Use `flush()` when shutting down to allow pending deliveries to finish.

This illustrates the basic pattern, not a complete production implementation. In production, also handle queue-full conditions, delivery errors, and shutdown timeouts.

---

## 4. What Can Go Wrong?

| Problem | What engineers should do |
|---|---|
| Local producer queue becomes full (`BufferError`) | Apply backpressure, process events, and retry with a deadline |
| Application exits before outstanding messages complete | Use bounded `flush()` during graceful shutdown |
| Broker rejects a message or delivery times out | Track delivery callbacks if reliability matters |
| Broker becomes unavailable | Use appropriate timeouts, retry policies, and monitoring |
| `flush()` returns zero | Understand that outstanding work finished, **not necessarily that every delivery succeeded** |

**Important:** Not calling `poll()` does not automatically mean the producer queue will fill. Background networking sends messages independently.

---

## 5. Fire-and-Forget vs Reliable Publishing

| Fire-and-forget | Delivery-tracked publishing |
|---|---|
| No individual delivery-result tracking | Registers delivery callbacks |
| Simpler application logic | Detects individual delivery failures |
| Appropriate when occasional loss is acceptable | Better for business-critical events |
| Still uses asynchronous sending | Also uses asynchronous sending |

Both patterns can use the same Kafka Producer API.

The key difference is whether your application records and acts on the delivery outcomes.

---

## 6. The Five Things to Memorize

1. **`produce()` queues:** It does not guarantee that Kafka received the message.
2. **Background networking sends:** Kafka's client library handles transmission automatically.
3. **`poll(0)` services events:** It does not fetch topic messages or initiate sending.
4. **`flush()` waits:** Call it before shutdown; check its return value.
5. **Fire-and-forget sacrifices delivery visibility:** Use callbacks and failure recovery when reliability matters.

### Final Memory Trick

**QUEUE → SEND → SERVICE → DRAIN**

- **QUEUE** — `produce()`
- **SEND** — Background networking
- **SERVICE** — `producer.poll(0)`
- **DRAIN** — `producer.flush(timeout)`

> **Engineering takeaway:** Reuse one producer, publish asynchronously, service events regularly, handle local queue pressure, and drain outstanding work during shutdown. If you must know whether messages arrived, use delivery callbacks rather than pure fire-and-forget.


### Answer without scrolling up

1. In one sentence, what does "fire-and-forget" mean, and what is the postcard analogy?
2. `produce()` returned successfully. Is the message on the broker yet? What actually sends it?
3. You ignore delivery results. Why must you *still* call `poll()` regularly? What two things does `poll()` do?
4. Why can forgetting `flush()` lose messages even when nothing crashed?
5. A message failed to deliver and your code printed no error. Is that a bug? Why or why not?
6. Give two message types where fire-and-forget is the right choice, and two where it is dangerous.
7. The repo's file has a `require_delivery_confirmation=True` mode. Why does that arguably not belong in a file called "fire and forget"?

### The pattern in one picture

```text
   produce()  ─┐
               ├─ repeat ──  (fast, no waiting)
   poll(0)   ─┘
   ...
   flush()        once, before exit   (don't lose what's queued)

   NO callback  =  NO idea if anything failed   ← this defines the pattern
```

**Next: Producer Pattern 2 — Callback-Confirmed.** We add one thing, a delivery callback, and the silent failure from Section 4 becomes a loud, catchable error.